# Pre-processing Cause of Deaths Data

This notebook focuses on pre-processing and consolidating multiple cause of death datasets from the Institute for Health Metrics and Evaluation (IHME) Global Burden of Disease (GBD) study. The goal is to create a clean, unified dataset for further analysis.

## 1. Import Required Libraries

We'll start by importing pandas, which is essential for data manipulation and analysis.

In [1]:
import pandas as pd

## 2. Load Datasets


Loading three separate CSV files from the IHME Global Burden of Disease (GBD) 2021 study on causes of death. Each file contains different segments of the complete dataset.

In [2]:
# Read four datasets into separate dataframes
df1 = pd.read_csv('../Datasets/new Dataset/IHME-GBD_2021_DATA-3bf79be6-1.csv')
df2 = pd.read_csv('../Datasets/new Dataset/IHME-GBD_2021_DATA-3bf79be6-2.csv')
df3 = pd.read_csv('../Datasets/new Dataset/IHME-GBD_2021_DATA-3bf79be6-3.csv')

## 3. Initial Data Exploration


Let's preview the first few rows from each dataset to understand their structure and content. This will help us verify that the data was loaded correctly and identify similarities or differences across datasets.

In [3]:
# Display the head of all 4 datasets side by side
pd.concat(
    [df1.head(), df2.head(), df3.head()],
    keys=['df1', 'df2', 'df3'],
    names=['Dataset']
)

measure_id measure_name  location_id location_name  sex_id  \
Dataset                                                                 
df1     0           1       Deaths           97     Argentina       3   
        1           1       Deaths           97     Argentina       3   
        2           1       Deaths           97     Argentina       3   
        3           1       Deaths           97     Argentina       3   
        4           1       Deaths           97     Argentina       3   
df2     0           1       Deaths           40  Turkmenistan       3   
        1           1       Deaths           40  Turkmenistan       3   
        2           1       Deaths           40  Turkmenistan       3   
        3           1       Deaths           40  Turkmenistan       3   
        4           1       Deaths           40  Turkmenistan       3   
df3     0           1       Deaths           50    Montenegro       3   
        1           1       Deaths           50    Montenegro       3   
        2           1       Deaths           50    Montenegro       3   
        3           1       Deaths           50    Montenegro       3   
        4           1       Deaths           50    Montenegro       3   

          sex_name  age_id  age_name  cause_id  \
Dataset                                          
df1     0     Both      22  All ages       395   
        1     Both      22  All ages       396   
        2     Both      22  All ages       399   
        3     Both      22  All ages       401   
        4     Both      22  All ages       402   
df2     0     Both      22  All ages       701   
        1     Both      22  All ages       703   
        2     Both      22  All ages       705   
        3     Both      22  All ages       707   
        4     Both      22  All ages       708   
df3     0     Both      22  All ages       432   
        1     Both      22  All ages       435   
        2     Both      22  All ages       438   
        3     Both      22  All ages       441   
        4     Both      22  All ages       444   

                                      cause_name  metric_id metric_name  year  \
Dataset                                                                         
df1     0                   Chlamydial infection          1      Number  1990   
        1                   Gonococcal infection          1      Number  1990   
        2  Other sexually transmitted infections          1      Number  1990   
        3                      Acute hepatitis A          1      Number  1990   
        4                      Acute hepatitis B          1      Number  1990   
df2     0           Poisoning by carbon monoxide          1      Number  2005   
        1               Poisoning by other means          1      Number  2005   
        2         Unintentional firearm injuries          1      Number  2005   
        3    Other exposure to mechanical forces          1      Number  2005   
        4   Adverse effects of medical treatment          1      Number  2005   
df3     0                        Cervical cancer          1      Number  2010   
        1                         Uterine cancer          1      Number  2010   
        2                        Prostate cancer          1      Number  2010   
        3                Colon and rectum cancer          1      Number  2010   
        4             Lip and oral cavity cancer          1      Number  2010   

                  val       upper       lower  
Dataset                                        
df1     0    2.960181    3.295362    2.695147  
        1    1.719927    1.966872    1.512458  
        2   10.173681   11.345633    9.253405  
        3   22.093804   45.118908   15.064646  
        4    9.600087   19.328814    5.731842  
df2     0   26.960845   29.243821   24.898377  
        1    4.396211    5.455654    2.581437  
        2   19.366235   24.856309   15.455301  
        3   43.902089   49.891991   37.423115  
        4   

### 3.1 Dataset Dimensions


Examining the shape (number of rows and columns) of each dataset to understand the overall size and structure.

In [4]:
# Display the shape of all 4 datasets in a table-like view
pd.DataFrame({
    'Dataset': ['df1', 'df2', 'df3'],
    'Rows': [df1.shape[0], df2.shape[0], df3.shape[0]],
    'Columns': [df1.shape[1], df2.shape[1], df3.shape[1]]
})

,Dataset,Rows,Columns
0,df1,500000,16
1,df2,500000,16
2,df3,467168,16


### 3.2 Analyzing Column Contents


This analysis examines key columns across all three datasets to compare:
1. The number of unique values in each column
2. The top three most common values

This helps identify consistent patterns, variations, and potential issues across datasets before merging them.

In [5]:
# For a more compact view just showing unique counts
compact_data = []
# List of columns to check for unique values and their counts
extra_columns = [
    'measure_id', 'measure_name', 'location_id',
    'sex_id', 'sex_name', 'age_id', 'age_name', 'cause_id',
    'metric_id', 'metric_name'
]
# Create a function to get unique values information for each column
def get_column_stats(df, column):
    unique_count = df[column].nunique()
    top_values = df[column].value_counts().head(3).index.tolist()
    return {'unique_count': unique_count, 'top_values': top_values}

# Collect stats for all datasets
results = {}

for col in extra_columns:
    results[col] = {
        'df1': get_column_stats(df1, col),
        'df2': get_column_stats(df2, col),
        'df3': get_column_stats(df3, col)
    }

# Create comparison dataframe
comparison_data = []

for col in extra_columns:
    row = {
        'Column': col,
        'df1_unique_count': results[col]['df1']['unique_count'],
        'df1_unique_value': str(results[col]['df1']['top_values']),
        'df2_unique_count': results[col]['df2']['unique_count'],
        'df2_unique_value': str(results[col]['df2']['top_values']),
        'df3_unique_count': results[col]['df3']['unique_count'],
        'df3_unique_value': str(results[col]['df3']['top_values'])
    }
    comparison_data.append(row)

# Create a clean DataFrame view
comparison_df = pd.DataFrame(comparison_data)

# Display the table
comparison_df

,Column,df1_unique_count,df1_unique_value,df2_unique_count,df2_unique_value,df3_unique_count,df3_unique_value
0,measure_id,1,[1],1,[1],1,[1]
1,measure_name,1,['Deaths'],1,['Deaths'],1,['Deaths']
2,location_id,204,"[106, 6, 111]",204,"[51, 49, 50]",204,"[133, 154, 155]"
3,sex_id,1,[3],1,[3],1,[3]
4,sex_name,1,['Both'],1,['Both'],1,['Both']
5,age_id,1,[22],1,[22],1,[22]
6,age_name,1,['All ages'],1,['All ages'],1,['All ages']
7,cause_id,232,"[357, 358, 353]",232,"[320, 319, 970]",232,"[1058, 1056, 1029]"
8,metric_id,1,[1],1,[1],1,[1]
9,metric_name,1,['Number'],1,['Number'],1,['Number']


To streamline the dataset and retain only relevant features for analysis, the following columns will be removed:

- measure_id: A unique identifier for the measure; not needed for analysis.
- measure_name: Constantly indicates “Death”, which is already known; redundant.
- location_id: A unique identifier for the location; not required.
- sex_id: A unique identifier for sex; unnecessary.
- sex_name: Always indicates “BOTH”, adding no analytical value.
- age_id: Corresponds to “All Ages”; not useful for detailed insights.
- cause_id: A unique identifier for the cause; not needed.
- metric_id: A unique identifier for the metric; irrelevant to the analysis.
- metric_name: Constantly reports “Number”, offering no variation or additional insight.

These columns do not contribute to the analytical goals and will be excluded from further processing.

## 5. Feature Selection

Following the data analysis, we're removing columns identified as redundant or unnecessary for our analytical goals. This includes:
- IDs (measure_id, location_id, sex_id, age_id, cause_id, metric_id)
- Constant values (measure_name="Death", sex_name="BOTH", metric_name="Number")
- Other non-informative fields

This streamlines the dataset for more focused analysis.

In [6]:
df1 = df1.drop(columns=extra_columns)
df2 = df2.drop(columns=extra_columns)
df3 = df3.drop(columns=extra_columns)

### 5.1 Verifying Dataset Dimensions After Column Removal

Checking the shape of each dataset after removing unnecessary columns to verify the change in structure.

In [7]:
print("df1 shape:", df1.shape)
print("df2 shape:", df2.shape)
print("df3 shape:", df3.shape)

df1 shape: (500000, 6)
df2 shape: (500000, 6)
df3 shape: (467168, 6)


### 5.2 Previewing Cleaned Datasets


Displaying the first few rows of each dataset after column removal to verify the data structure is as expected.

In [8]:
# Display the head of all 4 datasets side by side
pd.concat(
    [df1.head(), df2.head(), df3.head()],
    keys=['df1', 'df2', 'df3'],
    names=['Dataset']
)

location_name                             cause_name  year  \
Dataset                                                                
df1     0     Argentina                   Chlamydial infection  1990   
        1     Argentina                   Gonococcal infection  1990   
        2     Argentina  Other sexually transmitted infections  1990   
        3     Argentina                      Acute hepatitis A  1990   
        4     Argentina                      Acute hepatitis B  1990   
df2     0  Turkmenistan           Poisoning by carbon monoxide  2005   
        1  Turkmenistan               Poisoning by other means  2005   
        2  Turkmenistan         Unintentional firearm injuries  2005   
        3  Turkmenistan    Other exposure to mechanical forces  2005   
        4  Turkmenistan   Adverse effects of medical treatment  2005   
df3     0    Montenegro                        Cervical cancer  2010   
        1    Montenegro                         Uterine cancer  2010   
        2    Montenegro                        Prostate cancer  2010   
        3    Montenegro                Colon and rectum cancer  2010   
        4    Montenegro             Lip and oral cavity cancer  2010   

                  val       upper       lower  
Dataset                                        
df1     0    2.960181    3.295362    2.695147  
        1    1.719927    1.966872    1.512458  
        2   10.173681   11.345633    9.253405  
        3   22.093804   45.118908   15.064646  
        4    9.600087   19.328814    5.731842  
df2     0   26.960845   29.243821   24.898377  
        1    4.396211    5.455654    2.581437  
        2   19.366235   24.856309   15.455301  
        3   43.902089   49.891991   37.423115  
        4   28.999787   32.747448   25.935861  
df3     0   25.629488   31.187440   20.681259  
        1   15.985706   21.156587   12.131854  
        2   83.087657  110.968458   65.881866  
        3  143.454558  165.333431  124.411804  
        4   18.088328   21.263567   15.390206

## 6. Data Consolidation


Now that we've cleaned and verified each dataset, we'll concatenate all three datasets into a single comprehensive DataFrame. This creates a unified dataset for subsequent analysis.

In [9]:
# Concatenate the four datasets into a single DataFrame
df_all = pd.concat([df1, df2, df3], ignore_index=True)

### 6.1 Verifying Consolidated Dataset

Checking the dimensions of the combined dataset to ensure all records have been properly included.

In [10]:
print("df_all shape:", df_all.shape)

df_all shape: (1467168, 6)


## 7. Missing Value Detection and Handling:
Since the data was sourced from a very well reputed source, it was expected that there will be no rows containing any NAN

In [11]:
# Detect rows in the consolidated dataset that contain any missing (NA) values
missing_rows = df_all[df_all.isna().any(axis=1)]
print("total number of rows containing any NA value: ", len(missing_rows))

total number of rows containing any NA value:  0


## 8. Data Type Validation
Its important to check datatypes of each column before doing any 

In [12]:
# Check datatypes of each column in df_all and print as a dataframe
pd.DataFrame(df_all.dtypes, columns=['dtype']).reset_index().rename(columns={'index': 'column'})

,column,dtype
0,location_name,object
1,cause_name,object
2,year,int64
3,val,float64
4,upper,float64
5,lower,float64


## 8. Duplicate Detection

In [13]:
# Check for duplicates in df_all based on location_name, year, and cause_name
duplicate_rows = df_all[df_all.duplicated(subset=['location_name', 'year', 'cause_name'], keep=False)]
print("Total number of duplicate rows:", len(duplicate_rows))
duplicate_rows.head()

Total number of duplicate rows: 0


,location_name,cause_name,year,val,upper,lower


In [14]:
print(f"Dataset shape before dropping rows: {df_all.shape}")
df_all = df_all[df_all['cause_name'] != "Total Cancers excluding Non-melanoma skin cancer"]
print(f"Dataset shape after dropping rows: {df_all.shape}")

Dataset shape before dropping rows: (1467168, 6)
Dataset shape after dropping rows: (1460844, 6)


In [15]:
df_all = df_all.rename(columns={
    'location_name': 'Country',
    'cause_name': 'Cause',
    'year': 'Year',
    'val': 'Deaths',
    'upper': 'Deaths Upper',
    'lower': 'Deaths Lower'
})

print("Columns after renaming:", df_all.columns.tolist())

Columns after renaming: ['Country', 'Cause', 'Year', 'Deaths', 'Deaths Upper', 'Deaths Lower']


In [16]:
print(df_all['Deaths'] == max(df_all['Deaths']))

0          False
1          False
2          False
3          False
4          False
           ...  
1467163    False
1467164    False
1467165    False
1467166    False
1467167    False
Name: Deaths, Length: 1460844, dtype: bool


## 7. Export Processed Dataset


Saving the final consolidated and processed dataset to a CSV file for future analysis. This file will be the input for subsequent analytical work.

In [17]:
df_all.to_csv('all_cause_of_death_data.csv', index=False)

## Conclusion


In this notebook, we successfully:

1. Loaded three separate cause of death datasets from the IHME GBD study which were divided apparently because of the large length of dataset (~1.4 million)
2. Performed initial exploratory data analysis to understand the structure and content
3. Identified and removed redundant or unnecessary columns
4. Filtered the data to focus only on death-related records
5. Consolidated the three datasets into a single comprehensive dataset
6. Exported the final processed dataset to a CSV file for subsequent analysis

This pre-processed dataset (`all_cause_of_death_data.csv`) now provides a clean, unified source for analyzing global cause of death data.